# Práctica: Anonimización de Datos — Ataque de Enlace (Linkage Attack)

---

## 0. Carga del dataset Adult Census Income

In [2]:
import pandas as pd

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, names=COLUMNS, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Dataset cargado: {len(df)} registros, {df.shape[1]} atributos')
df.head()

Dataset cargado: 32561 registros, 15 atributos


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 1. Clasificación de atributos

### Identificadores directos
El dataset Adult Census Income **no contiene identificadores directos** (nombre, DNI, email, teléfono, pasaporte). Esta es una característica habitual de datasets publicados para investigación.

### Cuasi-identificadores (QI)
Son atributos que, combinados, pueden reidentificar a un individuo al cruzarse con fuentes externas:

| Atributo | Estrategias de anonimización |
|---|---|
| `age` | Generalización por rangos (20-30, 30-40...) · Supresión de edades extremas |
| `sex` | Supresión del atributo · Generalización a categoría única |
| `native_country` | Generalización geográfica (región/continente) · Supresión de países minoritarios |
| `race` | Supresión del atributo · Generalización a categoría 'Persona' |
| `education` | Generalización por niveles (básico/medio/superior) |
| `occupation` | Generalización por sectores |

### Atributos sensibles
Información privada cuya divulgación supone el riesgo principal:
- `income` (categoría de salario: **atributo sensible principal**)
- `capital_gain`, `capital_loss`
- `occupation`

### Atributos de baja sensibilidad / estadísticos
- `fnlwgt`, `education_num`, `marital_status`, `relationship`, `hours_per_week`

## 2. Implementación de k-anonymity (k=3)

### Técnicas aplicadas por atributo

| Atributo | Técnica | Justificación |
|---|---|---|
| `age` | **Generalización** por rangos de 10 años y **top-coding** a partir de 70 años | La edad exacta es altamente identificativa. Agruparla en intervalos (por ejemplo, `30-39`) reduce la granularidad sin perder excesiva utilidad analítica.  En edades avanzadas, donde existen menos registros en el dataset, se aplicó una generalización más amplia agrupando todas las edades iguales o superiores a 70 años en la categoría `70+`, reduciendo así la granularidad del dato y favoreciendo el cumplimiento de 3-anonymity. |
| `sex` | **Supresión total** | Este atributo presenta solo dos categorías posibles y aporta escasa utilidad analítica en este ejercicio. Suprimirlo completamente reduce significativamente la capacidad de reidentificación cuando se combina con otros cuasi-identificadores. |
| `native_country` | **Generalización geográfica** | Un país concreto puede identificar individuos poco frecuentes. Agrupar los países en regiones amplias (por ejemplo, `Europe`, `America`, `Asia`) mantiene información geográfica general al tiempo que reduce el riesgo de identificación individual. |

In [ ]:
def anonimizar_age(age: int):
    """Generalización por rangos de 10 años y supresión apartir de 70 años."""
    if age >= 70:
      return "70+"
    low = (int(age) // 10) * 10
    return f'{low}-{low + 9}'

def anonimizar_country(country):
    """Generalización geográfica"""
    europe = ['England', 'Germany', 'France', 'Italy', 'Portugal', 'Greece', 'Poland', 'Hungary']
    america = ['United-States', 'Canada', 'Mexico', 'Cuba', 'Ecuador']
    asia = ['India', 'China', 'Japan', 'Iran', 'Philippines', 'Vietnam']

    if country in europe:
        return 'Europe'
    elif country in america:
        return 'America'
    elif country in asia:
        return 'Asia'
    else:
        return 'Other'


def aplicar_k_anonymity(df: pd.DataFrame, k: int = 3):
    """
    Aplica k-anonymity sobre age, sex y native_country.
    Garantiza que cada combinación de QI aparece al menos k veces.
    """
    df_anon = df.copy()

    df_anon['sex'] = '*'

    df_anon['age'] = df_anon['age'].apply(anonimizar_age)

    df_anon['native_country'] = df_anon['native_country'].apply(anonimizar_country)

    return df_anon


df_anon = aplicar_k_anonymity(df, k=3)

qi = ['age', 'sex', 'native_country']
print(f'Registros originales  : {len(df)}')
print(f'Registros anonimizados: {len(df_anon)}')
df_anon[['age', 'sex', 'native_country', 'income']].head(10)

groups = df_anon.groupby(qi).size().reset_index(name='count')
print(groups.sort_values('count').head())
print("¿Cumple 3-anonymity?:", "SÍ" if (groups['count'] >= 3).all() else "NO")

Registros originales  : 32561
Registros anonimizados: 32561
      age sex native_country  count
2   10-19   *         Europe      7
25    70+   *           Asia      9
1   10-19   *           Asia     11
26    70+   *         Europe     14
27    70+   *          Other     15
¿Cumple 3-anonymity?: SÍ


## 3. ¿Es suficiente anonimizar solo un atributo?

Comprobaremos que anonimizar únicamente `age` o únicamente `native_country` es **insuficiente** para garantizar k-anonymity.

In [7]:
def verificar_k(df_test, qi, k=3, label=''):
    sizes = df_test.groupby(qi).size()
    violaciones = (sizes < k).sum()
    print(f'[{label}]')
    print(f'  Grupos QI totales             : {len(sizes)}')
    print(f'  Grupos con tamaño < {k} : {violaciones}')
    print(f'  k-anonymity garantizado        : {"SÍ" if violaciones == 0 else "NO"}')
    if violaciones > 0:
        print('  Ejemplos de grupos que no cumplen k-anonymity:')
        display(sizes[sizes < k].head(5).to_frame('count').reset_index())
    print()


# Solo age generalizado (native_country y sex intactos)
df_solo_age = df.copy()
df_solo_age['age'] = df_solo_age['age'].apply(anonimizar_age)
verificar_k(df_solo_age, qi=['age', 'sex', 'native_country'],
            label='Solo age generalizado')

# Solo native_country generalizado por continentes (age y sex intactos)
df_solo_country = df.copy()
df_solo_country['native_country'] = df_solo_country['native_country'].apply(anonimizar_country)
verificar_k(df_solo_country, qi=['age', 'sex', 'native_country'],
            label='Solo native_country generalizado')

[Solo age generalizado]
  Grupos QI totales             : 434
  Grupos con tamaño < 3 : 166
  k-anonymity garantizado        : NO
  Ejemplos de grupos que no cumplen k-anonymity:


,age,sex,native_country,count
0,10-19,Female,Cambodia,1
1,10-19,Female,Dominican-Republic,1
2,10-19,Female,El-Salvador,2
3,10-19,Female,Germany,1
4,10-19,Female,Guatemala,1



[Solo native_country generalizado]
  Grupos QI totales             : 470
  Grupos con tamaño < 3 : 91
  k-anonymity garantizado        : NO
  Ejemplos de grupos que no cumplen k-anonymity:


,age,sex,native_country,count
0,17,Female,Asia,2
1,17,Female,Europe,1
2,17,Male,Europe,1
3,18,Female,Asia,2
4,18,Female,Europe,1


### Conclusión
Al mantener la precisión de los demás cuasi-identificadores, siguen existiendo combinaciones que identifican a un único individuo (grupos de tamaño 1 o 2). Un atacante solo necesita conocer los atributos no anonimizados para cruzarlos con una fuente externa y reidentificar al sujeto. **Es imprescindible anonimizar conjuntamente todos los cuasi-identificadores.**

## 4. Ataque de enlace (Linkage Attack)

### Explicación teórica

Un atacante dispone de:
- **A)** El dataset anonimizado (edad en rangos, native_country generalizado, sexo suprimido).
- **B)** Un dataset externo con nombre, pasaporte, age exacto, sex y native_country obtenidos de fuentes públicas (redes sociales, registros electorales, bases de datos comerciales, etc.).

**Proceso del ataque:**
1. El atacante aplica las mismas transformaciones de generalización al dataset externo
2. Cruza (JOIN) el dataset externo con el anonimizado usando los cuasi-1identificadores como clave de cruce.
3. Cuando el cruce devuelve un único valor de `income` para un individuo, la reidentificación es inequívoca.

In [5]:
# Cargar dataset externo ficticio
ext = pd.read_csv('external_dataset.csv')
print('Dataset externo:')
display(ext)

# Aplicar las MISMAS transformaciones de anonimización al dataset externo
ext['sex'] = '*'
ext['age_range'] = ext['age'].apply(anonimizar_age)
ext['native_country_anon'] = ext['native_country'].apply(anonimizar_country)


print('\nDataset externo con QI anonimizados:')
display(ext[['fictitious_name', 'passport_id', 'age_range', 'sex', 'native_country_anon']])

Dataset externo:


,fictitious_name,passport_id,hours_per_week,education,native_country,sex,age,employer_or_position
0,Alex Carter,P730000,3,Bachelors,Hungary,Female,81,Walmart
1,Jordan Diaz,P730001,40,Bachelors,England,Female,90,Amazon
2,Taylor Foster,P730002,50,HS-grad,Ecuador,Male,90,Target
3,Casey Patel,P730003,15,1st-4th,Poland,Male,81,Costco
4,Morgan Reed,P730004,24,HS-grad,United-States,Female,90,The Home Depot
5,Riley Hayes,P730005,40,5th-6th,NaN,Female,70,US Postal Service Clerk
6,Avery Kim,P730006,28,10th,United-States,Female,82,State Government Program Analyst
7,Parker Brooks,P730007,41,Some-college,Vietnam,Male,71,City Administrative Assistant
8,Quinn Bennett,P730008,40,HS-grad,United-States,Male,90,IRS Revenue Agent
9,Skyler Morris,P730009,16,11th,South,Female,20,County Public Works Coordinator



Dataset externo con QI anonimizados:


,fictitious_name,passport_id,age_range,sex,native_country_anon
0,Alex Carter,P730000,70+,*,Europe
1,Jordan Diaz,P730001,70+,*,Europe
2,Taylor Foster,P730002,70+,*,America
3,Casey Patel,P730003,70+,*,Europe
4,Morgan Reed,P730004,70+,*,America
5,Riley Hayes,P730005,70+,*,Other
6,Avery Kim,P730006,70+,*,America
7,Parker Brooks,P730007,70+,*,Asia
8,Quinn Bennett,P730008,70+,*,America
9,Skyler Morris,P730009,20-29,*,Other


In [6]:
# Preparar dataset anonimizado para el cruce
df_anon_merge = df_anon[['age', 'sex', 'native_country', 'income']].copy()
df_anon_merge.columns = ['age_range', 'sex', 'native_country_anon', 'income']

# JOIN por cuasi-identificadores
merged = ext.merge(
    df_anon_merge,
    on=['age_range', 'sex', 'native_country_anon'],
    how='inner'
)

# Solo reidentificaciones inequívocas (el income es único en todos los matches)
reidentificados = (
    merged.groupby(['fictitious_name', 'passport_id'])
    .filter(lambda g: g['income'].nunique() == 1)
    .drop_duplicates(subset=['fictitious_name', 'passport_id'])
)

print('REGISTROS REIDENTIFICADOS INEQUÍVOCAMENTE:')
display(reidentificados[['fictitious_name', 'passport_id', 'age_range', 'sex', 'native_country_anon', 'income']])
print(f'\nTotal reidentificados: {len(reidentificados)} de {len(ext)} individuos en el dataset externo')

REGISTROS REIDENTIFICADOS INEQUÍVOCAMENTE:


,fictitious_name,passport_id,age_range,sex,native_country_anon,income
1830,Parker Brooks,P730007,70+,*,Asia,<=50K



Total reidentificados: 1 de 10 individuos en el dataset externo


## ¿Por qué falla k-anonymity en este escenario?

k-anonymity garantiza que dentro del dataset anonimizado, cada combinación
de cuasi-identificadores `(age_range, native_country_anon)` aparece en al
menos k=3 registros. Y eso se cumple — el grupo `(70+, Asia)` tiene 3 o
más personas en el dataset anonimizado.

El fallo está en que **k-anonymity protege el dataset publicado, pero no
controla la perspectiva del atacante**. Parker Brooks es el único individuo
del dataset externo que cae en el grupo `(70+, Asia)`. El atacante no
necesita romper el grupo — le basta con saber que de todas las personas que
conoce, solo una encaja en esa combinación de cuasi-identificadores. Eso
convierte la garantía de k=3 en irrelevante.

Nótese además que `sex` fue **suprimido** (`*`) precisamente para reducir
la granularidad y evitar grupos pequeños. Sin embargo el ataque sigue
funcionando: incluso sin `sex` como atributo discriminante, la combinación
de rango de edad y región geográfica es suficientemente específica para
reidentificar a Parker Brooks de forma inequívoca. Esto ilustra que
**suprimir atributos no es suficiente si los restantes siguen siendo
discriminantes** cuando se cruzan con un dataset externo.

Dicho de otro modo: k-anonymity asume que el atacante no tiene información
auxiliar sobre a qué grupo pertenece cada individuo. Esa suposición es
falsa en cuanto existe un dataset externo.

---

## ¿Lo habrían evitado l-diversity o t-closeness?

**l-diversity — no lo habría evitado.**

l-diversity exigiría que dentro del grupo `(70+, Asia)` hubiera al menos
l=2 valores distintos de income (`<=50K` y `>50K`). Eso protege contra el
caso en que todos los del grupo tienen el mismo income, pero no cambia nada
para Parker Brooks: el atacante ya sabe que es él quien está en ese grupo,
con independencia de cuántos valores de income haya dentro. Una vez
reidentificada la persona, l-diversity no aporta protección adicional.

l-diversity protege contra ataques de homogeneidad (cuando todos en el grupo tienen el mismo income), pero **no evita el linkage attack** cuando el cruce de cuasi-identificadores produce grupos pequeños en el espacio del atacante. Si el atacante tiene información de contexto adicional, puede seguir inferiendo el income.

**t-closeness — tampoco lo habría evitado.**

t-closeness exigiría que la distribución de income dentro del grupo
`(70+, Asia)` se parezca a la distribución global del dataset. Eso reduce
la información que el atacante puede inferir estadísticamente sobre el
income del grupo, pero no impide la reidentificación de la identidad.
Parker Brooks sigue siendo el único individuo del dataset externo en ese
grupo, y el atacante puede leer su income directamente del registro cruzado.

**La raíz del problema es estructural:** las tres técnicas operan sobre el
dataset publicado de forma aislada. Ninguna modela ni limita lo que el
atacante puede hacer con información externa. La única técnica que ofrece
garantías formales independientemente del conocimiento auxiliar del atacante
es la **privacidad diferencial**, que añade ruido matemáticamente calibrado
de forma que la presencia o ausencia de cualquier individuo concreto no sea
detectable.